In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

### Make a specific spatial region spatially unfair

**TODO.**

The idea is to partition the space with a uniform grid, select a specific cell, and then relabel the objects associated with that cell. This simulates a trajectory classifier that is unfair w.r.t. that cell's objects.

In [ ]:
import pickle

# Materialize a grid over the geographical area in which the objects move.
#
# TODO: at the moment we are using one of the grids materialized in the notebook "users-to-cells mapping" (as such,
#       this grid has already a set of candidates (subset of cells) generated from the notebook "Subset of cells to test"),
#       and subsequently used among the grids employed in the spatial scan statistics.
#       In the future, we'll have to (1) use a spatial partition not used in the hyp.test., (2) whose cells have a different
#       shape than the squared ones used for the materialized grids, and (3) we'll have to somehow associate the objects to
#       the subsets of cells of this new grid.

# Read the candidates generated for a given grid, in their original format.
grid_resolution = 50
grid_offset = 20
path_grid_candidates = f'./data_simulator/huge_dataset/gencand/candidates_{grid_resolution}_{grid_offset}.pkl'
with open(path_grid_candidates, "rb") as f:
    grid_candidates = pickle.load(f)


# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

In [ ]:
# For each candidate, compute the number of cells and associated objects.
grid_candidates['num_objs'] = grid_candidates['list_users'].map(len)
grid_candidates['num_cells'] = grid_candidates.index.map(lambda x: 1 if isinstance(x, int) else len(x))

# Select the candidates with just one cell.
sel_candidates = grid_candidates.loc[grid_candidates['num_cells'] == 1].copy()

# Now pick a candidate for which we want to penalize the objects.
sel_candidates.sort_values('num_objs', ascending=False, inplace = True)
sel_candidates.head(20)

In [ ]:
# 1 - Generate the "fair" labels for a given number of objects. 
n_objects = 100000
positive_rate = 0.6
labels = np.random.default_rng().binomial(n=1, p=positive_rate, size=n_objects).astype(np.int8)


# 2 - Now, generate the unfair labels to be applied to a selected set of objects.
penalized_candidate = 3678
ids_penalized_objects = sel_candidates.loc[penalized_candidate, 'list_users']
num_penalized_objects = ids_penalized_objects.size
penalized_positive_rate = 0.5
penalized_labels_obj = np.random.default_rng().binomial(n=1, p=penalized_positive_rate, size=num_penalized_objects).astype(np.int8)


# Apply the unfair labels to the objects associated with 'penalized candidate'.
final_labels = labels.copy()
final_labels[ids_penalized_objects] = penalized_labels_obj

### Write the synthetic unfair labels to disk

In [ ]:
print(labels.sum() / labels.size, final_labels.sum() / final_labels.size)
dict_unfair_dataset = {'labels': final_labels, 'obj_ids' : ids_penalized_objects}

path_unfair_dataset = './experiments/unfair_dataset.pkl'
with open(path_unfair_dataset, "wb") as f:
    pickle.dump(dict_unfair_dataset, f)